# อ่านสลิปด้วย Roboflow (หาช่อง) + EasyOCR (อ่านข้อความ)

Flow: ภาพสลิป → โมเดลที่เทรนเองบน Roboflow หากล่อง (`money`, `name`, `sent`, `bank`, `type payment`, `memo`, `silp`)
→ ตัดภาพแต่ละกล่อง → EasyOCR (th+en) → แปลงเป็นข้อมูล (ยอด ชื่อ วันที่ รหัสอ้างอิง)



In [ ]:
!pip -q install easyocr rfdetr

In [ ]:
import base64, glob, gzip, hashlib, json, re, time, urllib.parse, urllib.request
from collections import Counter
from pathlib import Path

import cv2
import easyocr
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from rfdetr import RFDETRSmall

# ---------------- config ----------------
IMAGE_DIR = Path("/kaggle/input/datasets/chujikgch/887uu7/slip - Copy")          # <- โฟลเดอร์รูปสลิป
N_IMAGES = 261                                   # ทดสอบกี่รูป (None = ทุกรูป) ลองเร็วๆ ใช้ 20
CONFIDENCE = 40                                 # % ความมั่นใจต่ำสุดของกล่อง
PAD = 6                                         # เผื่อขอบก่อนตัด (พิกเซล)
USE_TROCR = True                                # True = อ่านชื่อซ้ำด้วย ThaiTrOCR (ต้องมี GPU ถึงจะเร็ว)
COMBINE = "smart"
ROBOFLOW_MODEL = "own-rfdetr-small"             # แค่ป้ายบันทึกลงตาราง (ไม่ได้เรียก Roboflow แล้ว)
USE_DETECT_CACHE = False                        # ไม่ใช้แคชแล้ว (เซลล์รันหลักยังอ้างตัวแปรนี้)
ALLOW_API_FALLBACK = False

# ---- ThaiTrOCR ที่ fine-tune แล้ว: หาโฟลเดอร์ที่มีทั้ง config.json + model.safetensors ใต้ /kaggle/input (ลึกกี่ชั้นก็ได้) ----
_cands = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/model.safetensors", recursive=True)
                 if (Path(p).parent / "processor_config.json").exists() or (Path(p).parent / "preprocessor_config.json").exists()})
_cands = [c for c in _cands if "detectorlabel" not in c]                     # กันไปเจอโฟลเดอร์อื่น
TROCR_MODEL = next((c for c in _cands if "finetune" in c.lower() or "trocr" in c.lower()), _cands[0] if _cands else "openthaigpt/thai-trocr")
print("ThaiTrOCR:", TROCR_MODEL, "(fine-tune แล้ว)" if _cands else "(ตัวเดิม: ไม่พบโมเดลที่ fine-tune ใน Input)")

# ---- โมเดลตรวจจับที่เทรนเอง (dataset detectorlabel) ----
WEIGHTS = (sorted(glob.glob("/kaggle/input/**/checkpoint_best_total.pth", recursive=True)) or [None])[0]
assert WEIGHTS, "ไม่พบ checkpoint_best_total.pth: Add Input ของ dataset detectorlabel ก่อน"
DET_DIR = Path(WEIGHTS).parent
ID2NAME = {int(k): v for k, v in json.load(open(DET_DIR / "id2name.json")).items()}
SPLIT = json.load(open(DET_DIR / "split_of.json")) if (DET_DIR / "split_of.json").exists() else {}   # md5 ของรูป -> train/valid/test
det_model = RFDETRSmall(pretrain_weights=WEIGHTS)
print("น้ำหนัก:", WEIGHTS, "| class:", ID2NAME, "| split:", Counter(SPLIT.values()))

# class ในโมเดล -> ช่องที่ใช้
ALIAS = {
    "money": "amount",
    "name": "from_name",
    "sent": "to_name",
    "bank": "from_bank",
    "type payment": "title",
    "memo": "memo",
}
NAME_FIELDS = {"from_name", "to_name"}

reader = easyocr.Reader(["th", "en"], gpu=True)   # ครั้งแรกจะโหลดโมเดล ~1 นาที
_dev = "cuda" if torch.cuda.is_available() else "cpu"
if USE_TROCR:
    trocr_proc = TrOCRProcessor.from_pretrained(TROCR_MODEL)
    trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL).to(_dev).eval()
_trocr_errors = []                                # ล้าง error ของรอบก่อน
print("พร้อม | TrOCR", ("บน " + _dev) if USE_TROCR else "ปิด")

In [ ]:
_cache_miss = []                                  # เซลล์รันหลักยังอ้างตัวแปรนี้ (ตอนนี้ไม่ได้ใช้แคช จึงว่างเสมอ)


def detect(path, conf=None):
    # ตรวจจับกล่องด้วยโมเดลที่เทรนเอง (RF-DETR รันในเครื่อง ไม่เรียก API) คืนรูปแบบเดียวกับที่ Roboflow ให้
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    det = det_model.predict(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), threshold=(conf or CONFIDENCE) / 100)
    preds = [{"class": ID2NAME.get(int(c), str(c)), "confidence": float(s), "x": (b[0] + b[2]) / 2, "y": (b[1] + b[3]) / 2,
              "width": b[2] - b[0], "height": b[3] - b[1]}
             for b, c, s in zip(det.xyxy, det.class_id, det.confidence)]
    return preds, {"width": w, "height": h}


def crop_box(img, p, scale):
    h, w = img.shape[:2]
    cx, cy, bw, bh = p["x"] * scale, p["y"] * scale, p["width"] * scale, p["height"] * scale
    x0 = max(int(cx - bw / 2) - PAD, 0)
    y0 = max(int(cy - bh / 2) - PAD, 0)
    x1 = min(int(cx + bw / 2) + PAD, w)
    y1 = min(int(cy + bh / 2) + PAD, h)
    crop = img[y0:y1, x0:x1]
    if crop.size and crop.shape[0] < 96:           # ช่องเตี้ยเกินไป ขยายให้ OCR อ่านง่าย
        f = 96 / crop.shape[0]
        crop = cv2.resize(crop, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    return crop


def read_amount(crop):
    # width_ths ต่ำ = แยกเป็นคำ ไม่ให้ป้าย "จำนวน" (ที่ allowlist บังคับให้เป็นเลข) ไปติดกับตัวเลขยอดเงิน
    if crop is None or crop.size == 0:
        return None, None
    rows_d = reader.readtext(crop, detail=1, paragraph=False, allowlist="0123456789.,", width_ths=0.1)
    rows_n = reader.readtext(crop, detail=1, paragraph=False, width_ths=0.1)        # อ่านปกติ (ไม่บังคับเป็นตัวเลข) ไว้เทียบ
    return pick_amount(rows_d), pick_amount(rows_n)


def ocr_text(crop, allowlist=None):
    # อ่านทุกบรรทัดในกรอบ เรียงจากบนลงล่าง แล้วต่อด้วยช่องว่าง
    if crop is None or crop.size == 0:
        return ""
    rows = reader.readtext(crop, detail=1, paragraph=False, allowlist=allowlist)
    items = []
    for box, text, _conf in rows:
        ys = [pt[1] for pt in box]
        xs = [pt[0] for pt in box]
        if text.strip():
            items.append({"yc": sum(ys) / 4, "h": max(ys) - min(ys), "x": min(xs), "t": text.strip()})
    items.sort(key=lambda i: i["yc"])
    lines = []                                   # จัดกลุ่มเป็นบรรทัดก่อน (กล่องที่อยู่แนวเดียวกัน) แล้วค่อยเรียงซ้าย->ขวา
    for it in items:
        if lines and abs(it["yc"] - lines[-1][0]["yc"]) < 0.6 * max(it["h"], lines[-1][0]["h"]):
            lines[-1].append(it)
        else:
            lines.append([it])
    return " ".join(" ".join(i["t"] for i in sorted(line, key=lambda i: i["x"])) for line in lines)


LABEL_TOKEN = re.compile(r"^(?:จาก|ไปยัง|ถึง|from|to)$", re.I)
MASK_OR_NUM = re.compile(r"^(?:[xX*]+[xX*0-9\-]*|[0-9๐-๙][0-9๐-๙\-,.]*)$")     # เลขบัญชีที่ปิดบัง / ตัวเลขล้วน
_trocr_errors = []


def name_line_image(crop):
    # ThaiTrOCR อ่านได้ทีละบรรทัด: ใช้ EasyOCR หา "บรรทัดชื่อ" (ข้ามป้ายจาก/ไปยัง/ถึง และเลขบัญชี) แล้วตัดภาพเฉพาะบรรทัดนั้น
    rows = reader.readtext(crop, detail=1, paragraph=False)
    items = []
    for box, text, _conf in rows:
        t = text.strip()
        if not t or LABEL_TOKEN.match(t) or MASK_OR_NUM.match(t):
            continue
        ys = [pt[1] for pt in box]
        xs = [pt[0] for pt in box]
        items.append({"yc": sum(ys) / 4, "h": max(ys) - min(ys), "x0": min(xs), "y0": min(ys), "y1": max(ys), "t": t})
    items.sort(key=lambda i: i["yc"])
    lines = []
    for it in items:
        if lines and abs(it["yc"] - lines[-1][0]["yc"]) < 0.6 * max(it["h"], lines[-1][0]["h"]):
            lines[-1].append(it)
        else:
            lines.append([it])
    for line in lines:                                           # บรรทัดแรกที่มีตัวอักษรไทย/อังกฤษอย่างน้อย 2 ตัว
        if len(re.findall(r"[A-Za-z\u0e00-\u0e7f]", " ".join(i["t"] for i in line))) >= 2:
            h, w = crop.shape[:2]
            y0 = max(int(min(i["y0"] for i in line)) - 4, 0)
            y1 = min(int(max(i["y1"] for i in line)) + 4, h)
            x0 = max(int(min(i["x0"] for i in line)) - 6, 0)
            return crop[y0:y1, x0:w]                             # ไปจนสุดขอบขวา กันคำที่ EasyOCR ตกหล่น
    return crop


def read_name_trocr(crop):
    # คืนข้อความจาก ThaiTrOCR (สตริงว่างถ้าผิดพลาด เก็บข้อผิดพลาดไว้ใน _trocr_errors)
    if not USE_TROCR or crop is None or crop.size == 0:
        return ""
    try:
        line = name_line_image(crop)
        pil = Image.fromarray(cv2.cvtColor(line, cv2.COLOR_BGR2RGB))
        with torch.no_grad():
            px = trocr_proc(images=pil, return_tensors="pt").pixel_values.to(_dev)
            ids = trocr_model.generate(px, max_length=64)
        return trocr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
    except Exception as err:
        _trocr_errors.append(repr(err)[:200])
        return ""

In [ ]:
THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
MONEY_RE = r"\d[\d,]*[.,]\d{2}(?!\d)"          # ตัวเลขที่ลงท้ายทศนิยม 2 ตำแหน่ง


def pick_amount(rows):
    # rows = ผล readtext (box, text, conf) ที่แยกเป็นคำแล้ว: เลือกคำที่เป็นยอดเงินจริง (มีทศนิยม 2 ตำแหน่ง)
    # ถ้ามีหลายคำ (เช่น ยอดใหญ่ กับ "0.00 ค่าธรรมเนียม") เลือกตัวที่ตัวอักษรสูงสุด ถ้าเท่ากันเลือกตัวที่อยู่ขวาสุด
    hits = []
    for box, text, _conf in rows:
        t = text.strip(" ,").translate(THAI_DIGITS)
        if re.fullmatch(MONEY_RE, t):
            ys = [pt[1] for pt in box]
            xs = [pt[0] for pt in box]
            hits.append((max(ys) - min(ys), max(xs), t))
    return max(hits)[2] if hits else None


STRICT_MONEY = r"(?:\d{1,3}(?:,\d{3})+|\d+)\.\d{2}"   # คอมมาต้องคั่นทีละ 3 หลัก เช่น 1,380.30 (1741,380.30 ไม่ผ่าน)


def choose_amount(digits_txt, normal_txt):
    # เลือกยอดจาก 2 วิธีอ่าน: ใช้ตัวที่รูปแบบถูกต้องก่อน (ตัวเลขล้วนมาก่อน) และติดธงเมื่อสองวิธีให้ค่าต่างกัน/รูปแบบผิด
    cands = [t for t in (digits_txt, normal_txt) if t]
    good = [t for t in cands if re.fullmatch(STRICT_MONEY, t)]
    pick = (good or cands or [None])[0]
    flag = ""
    if len({parse_money(t) for t in good}) > 1:
        flag = "conflict"
    elif cands and not good:
        flag = "invalid_format"
    return pick, flag


def parse_money(text):
    text = (text or "").translate(THAI_DIGITS)
    m = re.findall(MONEY_RE, text)
    if m:
        whole, dec = re.split(r"[.,](?=\d{2}$)", m[-1])
        return float(whole.replace(",", "").replace(".", "") + "." + dec)
    digits = re.sub(r"[^0-9]", "", text)
    return float(digits) if digits else None


import difflib

PREFIX = r"(?:นาย|นางสาว|นาง|น\.ส\.|ด\.ช\.|ด\.ญ\.)"
MASK_FIX = True          # สลิปที่ปิดนามสกุลเหลือ "แ." (SCB ชุดนี้): OCR อ่านเป็น l. / ii. / ศ. / โ. -> แก้กลับเป็น "แ." (เป็นกฎเฉพาะข้อมูลชุดนี้ ปิดได้)


def clean_name(text):
    s = (text or "").replace("ฺ", "")   # พินทุ ฺ ที่ OCR ใส่เกินมา (เช่น "ฺป")
    s = re.sub(r"([\u0e48-\u0e4b])\1+", r"\1", s.replace("\u0e4d\u0e32", "\u0e33")) 
    s = re.sub(r"\s*\(\s*\d+\s*\)\s*$", "", s)                          # รหัสท้ายชื่อ (24376)
    s = re.sub(r"^[.\s]+", "", s)                                      # จุดนำหน้าที่ OCR เติม เช่น ". teeraphat"
    s = re.sub(r"^(?:จาก|ลาก|ไปยัง|ถึง|from|to)\s*[:：]?\s+", "", s.strip(), flags=re.I)   # ป้ายนำหน้า ("ลาก" = "จาก" ที่ OCR อ่านผิด)
    s = re.sub(r"(?:(?<=\s)|^)(?:จาก|ไปยัง|ถึง)(?=\s|$)", " ", s)             # ป้ายที่ OCR วางไว้กลาง/ท้ายข้อความ (กรอบทั้งแถวของ SCB)
    s = re.sub(r"(?:(?<=\s)|^)[xX*]+[xX*0-9\-]*(?=\s|$)", " ", s)            # เลขบัญชีที่ปิดบัง เช่น xxx-xxx905-1, x-9660
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+[,;|]+\s*", " ", s).strip()                          # เครื่องหมายลอยที่ OCR เติมเกิน เช่น "วัดจันทาราม ,(เงิน...)" (ไม่แตะ "CO.,LTD.")
    # ตัดตั้งแต่คำบอกชนิดเลข (เลขบัญชี/ผู้เสียภาษี/biller id/รหัสร้านค้า) หรือเลขยาว 6 หลักขึ้นไปเป็นต้นไป
    s = re.sub(r"(?:(?<=\s)|^)(?:เลขประจำตัว|เลขที่|หมายเลข|เลขบัญชี|เลขอ้างอิง|รหัสอ้างอิง|รหัสร้านค้า|รหัสร้าน|biller|ร้านค้า|merchant id)\S*.*$|(?:(?<=\s)|^)[0-9๐-๙][0-9๐-๙\-]{5,}.*$", "", s, flags=re.I).strip()
    m = re.fullmatch(r"(.+?)\s+(%s)" % PREFIX, s)                     # คำนำหน้ามาอยู่ท้ายชื่อ -> ย้ายไปไว้หน้า
    if m:
        s = "%s %s" % (m.group(2), m.group(1))
    if MASK_FIX:
        s = re.sub(r"^(.*\S)\s+(?:l|ll|i|ii|I|II|\||ศ|โ)\.?$", r"\1 แ.", s)
    return s or None


def _norm_name(s):
    s = re.sub(r"^(?:นางสาว|นาง|นาย|น\.ส\.|mr\.?|mrs\.?|ms\.?)\s*", "", (s or "").replace("...", "").strip(), flags=re.I)
    return re.sub(r"[\s.*]", "", s).lower()


def _is_latin(s):
    s = s or ""
    return len(re.findall(r"[A-Za-z]", s)) > len(re.findall(r"[\u0e00-\u0e7f]", s))


def combine_names(easy, trocr, field=None):
    # รวมผลชื่อจาก EasyOCR กับ ThaiTrOCR ตาม COMBINE (ทั้งคู่ผ่าน clean_name แล้ว)
    # COMBINE: "smart" | "trocr_first" | "easy_first" | "trocr" | "easy" หรือ dict รายช่อง เช่น {"from_name": "smart", "to_name": "smart"}
    mode = COMBINE.get(field, "smart") if isinstance(COMBINE, dict) else COMBINE
    if mode == "easy":
        return easy
    if mode == "trocr":
        return trocr
    if mode == "smart":
        if not easy or not trocr:
            return easy or trocr
        if "ฯ" in trocr:
            return easy                                   # TrOCR ตัดข้อความยาวเป็น "ฯลฯ"
        if difflib.SequenceMatcher(None, _norm_name(easy), _norm_name(trocr)).ratio() < 0.35:
            return easy                                   # สองตัวแทบไม่เหมือนกัน = TrOCR อาจแต่งคำ -> เชื่อ EasyOCR
        return trocr                                      # ThaiTrOCR ที่ fine-tune แล้วอ่านวรรณยุกต์/สระแม่นกว่า -> ใช้เป็นหลัก
    first, second = (trocr, easy) if mode == "trocr_first" else (easy, trocr)
    return first or second


def clean_bank(text):
    s = re.sub(r"\s+", " ", (text or "").strip(" .")).strip()
    m = re.fullmatch(r"(.+?)\s+ธ", s)                                   # "กสิกรไทย ธ." -> "ธ.กสิกรไทย"
    if m:
        s = "ธ." + m.group(1).strip(" .")
    return s or None


def extract_date(text):
    t = (text or "").translate(THAI_DIGITS)
    m = re.search(r"(\d{1,2}\s*[ก-ฮ][ก-ฮ.]{1,10}\s*(?:25\d{2}|\d{2}))(?:\s*[-–]?\s*(\d{1,2}:\d{2}))?", t)
    if not m:
        m = re.search(r"(\d{1,2}[/.\-]\d{1,2}[/.\-]\d{2,4})(?:\s*(\d{1,2}:\d{2}))?", t)
    return " ".join(g for g in (m.group(1), m.group(2)) if g) if m else None


def extract_ref(text):
    m = re.search(r"อ้างอิง\s*[:：]?\s*([A-Za-z0-9]{8,})", text or "")
    if m:
        return m.group(1)
    for tok in re.split(r"\s+", text or ""):                              # ไม่ใช้เลขล้วน กันชนเบอร์โทร
        if re.fullmatch(r"[A-Za-z0-9]{12,}", tok) and re.search(r"[A-Za-z]", tok) and re.search(r"\d", tok):
            return tok
    return None

In [ ]:
OCR_DIGITS = str.maketrans({"o": "0", "O": "0", "D": "0", "l": "1", "I": "1", "|": "1", "S": "5", "B": "8"})   # ตัวอักษรที่ OCR สับสนกับตัวเลข
AMOUNT_PADS = (6, 40)                                  # เผื่อขอบซ้าย-ขวา (พิกเซล) อ่าน 2 ขนาด กันเลขหลักหน้าถูกตัด/ติดป้าย "จำนวน"


def _joined_line(rows):
    # ต่อคำในกรอบตามแนวนอน: OCR มักแยก "3" กับ "00" (จุดทศนิยมหาย) หรือ "0." กับ "50" -> ต่อกลับเป็น "3.00" / "0.50"
    items = sorted(rows, key=lambda r: min(pt[0] for pt in r[0]))
    text, boxes = "", []
    for box, t, _c in items:
        t = t.strip().translate(THAI_DIGITS)
        if not t:
            continue
        cur = t.translate(OCR_DIGITS)
        if text and re.fullmatch(r"\d{2}", cur) and re.search(r"\d$", text.translate(OCR_DIGITS)):
            text += "." + t
        elif text and text.endswith("."):
            text += t
        else:
            text += (" " if text else "") + t
        boxes.extend(box)
    return [(boxes, text, None)] if text else []


def money_candidates(rows):
    # ดึงยอดที่รูปแบบถูกต้องจากผล readtext: แก้ o->0, l->1 เฉพาะในก้อนที่เป็นตัวเลข แล้วหา "1,234.56" ในข้อความ
    out = []
    for box, text, _conf in list(rows) + _joined_line(rows):
        for tok in re.findall(r"[\d.,oOlI|SBD]+", text.translate(THAI_DIGITS)):
            tok = tok.translate(OCR_DIGITS).strip(",.")
            if not re.search(r"\d", tok):
                continue
            for m in re.finditer(r"(?<![\d,])" + STRICT_MONEY + r"(?!\d)", tok):
                v = m.group(0)
                if re.match(r"0\d", v):                     # "040.00" = เลขหลักหน้าหาย ไม่เชื่อ
                    continue
                ys = [pt[1] for pt in box]
                out.append((max(ys) - min(ys), v))
    return out


def read_amount_robust(img, p, scale):
    # อ่านยอด 2 ขนาดขอบ x 2 วิธี (ปกติ / ตัวเลขล้วน) แล้วโหวต: คืน (ข้อความยอด, flag)
    votes = {}
    for padx in AMOUNT_PADS:
        h, w = img.shape[:2]
        cx, cy, bw, bh = p["x"] * scale, p["y"] * scale, p["width"] * scale, p["height"] * scale
        x0, x1 = max(int(cx - bw / 2) - padx, 0), min(int(cx + bw / 2) + padx, w)
        y0, y1 = max(int(cy - bh / 2) - PAD, 0), min(int(cy + bh / 2) + PAD, h)
        crop = img[y0:y1, x0:x1]
        if crop.size and crop.shape[0] < 96:
            f = 96 / crop.shape[0]
            crop = cv2.resize(crop, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
        for kw, weight in (({}, 2), ({"allowlist": "0123456789.,"}, 1)):   # อ่านปกติน่าเชื่อกว่า (ตัวเลขล้วนชอบรวมป้าย "จำนวน" เป็นเลข)
            rows = reader.readtext(crop, detail=1, paragraph=False, width_ths=0.1, **kw)
            cands = money_candidates(rows)
            if cands:
                v = max(cands)[1]                         # ตัวอักษรสูงสุด = ยอดหลัก (ไม่ใช่ค่าธรรมเนียม 0.00)
                key = float(v.replace(",", ""))
                votes[key] = votes.get(key, 0) + weight
    if not votes:
        return None, "no_amount"
    ranked = sorted(votes.items(), key=lambda kv: -kv[1])
    best = ranked[0][0]
    flag = "" if len(ranked) == 1 else ("conflict" if ranked[1][1] >= ranked[0][1] * 0.5 else "minor_conflict")
    return "%.2f" % best, flag

In [ ]:
RETRY_CONF = 20          # % ถ้ากล่อง amount/from_name/to_name หายที่ CONFIDENCE ปกติ ให้ลองใหม่ที่ความมั่นใจต่ำนี้ (เติมเฉพาะช่องที่หาย) None = ปิด


def read_slip(path, full_fallback=True):
    img = cv2.imread(str(path))
    if img is None:
        return {"file": Path(path).name, "error": "อ่านไฟล์ไม่ได้"}, {}
    preds, size = detect(path)
    scale = img.shape[1] / size["width"] if size.get("width") else 1.0

    groups = {}
    for p in preds:
        field = ALIAS.get(p["class"], p["class"])
        if field != "silp":                                   # silp = กรอบทั้งใบ ไม่มีข้อความให้อ่าน
            groups.setdefault(field, []).append(p)

    need = {"amount", "from_name", "to_name"}
    missing = need - set(groups)
    retried = []
    if RETRY_CONF and missing:                                # ช่องสำคัญหาย -> ตรวจใหม่ที่ความมั่นใจต่ำ เติมเฉพาะช่องที่หาย
        for p in detect(path, RETRY_CONF)[0]:
            field = ALIAS.get(p["class"], p["class"])
            if field in missing:
                groups.setdefault(field, []).append(p)
        retried = sorted(missing & set(groups))

    texts, crops = {}, {}
    amount_flag, amount_alt = "", None
    for field, boxes in groups.items():
        if field in NAME_FIELDS and field not in retried:     # ชื่อ (ปกติ): ลองกล่องที่กว้างกว่าสูงก่อน / ช่องที่ตรวจซ้ำ: เรียงตามความมั่นใจ
            boxes = sorted(boxes, key=lambda b: -b["width"] / max(b["height"], 1))
        else:
            boxes = sorted(boxes, key=lambda b: -b["confidence"])
        for p in boxes[:3]:
            crop = crop_box(img, p, scale)
            if field == "amount":
                text, amount_flag = read_amount_robust(img, p, scale)
                amount_alt = None
                text = text or ""
                             # ไม่เจอคำที่เป็นยอดเงิน -> ถอยไปอ่านข้อความปกติ
            else:
                text = ocr_text(crop)
            if len(text.strip()) >= 2:
                texts[field], crops[field] = text, crop
                break

    # ไม่มีกล่องวันที่/รหัสอ้างอิง: ลองหาจากกล่อง type payment ก่อน แล้วค่อยอ่านทั้งใบ
    blob = " ".join(texts.get(k, "") for k in ("title", "amount"))
    date, ref = extract_date(blob), extract_ref(blob)
    if full_fallback and not (date and ref):
        whole = ocr_text(img)
        date, ref = date or extract_date(whole), ref or extract_ref(whole)

    names = {}                                                    # (EasyOCR, ThaiTrOCR, ผลรวม) ของแต่ละช่องชื่อ
    for f in ("from_name", "to_name"):
        easy = clean_name(texts.get(f))
        tro = clean_name(read_name_trocr(crops.get(f))) if USE_TROCR else None
        names[f] = (easy, tro, combine_names(easy, tro, f))

    row = {
        "file": Path(path).name,
        "folder": Path(path).parent.name,
        "amount": parse_money(texts.get("amount")),
        "amount_raw": texts.get("amount"),
        "amount_alt": amount_alt,                     # ผลจากอีกวิธีอ่าน ไว้เทียบ
        "amount_flag": amount_flag,                   # conflict = สองวิธีอ่านไม่ตรงกัน / invalid_format = รูปแบบผิด ควรตรวจ
        "from_name": names["from_name"][2],
        "from_name_easy": names["from_name"][0],      # ผล EasyOCR ล้วน ไว้เทียบ
        "from_name_trocr": names["from_name"][1],     # ผล ThaiTrOCR ล้วน
        "to_name": names["to_name"][2],
        "to_name_easy": names["to_name"][0],
        "to_name_trocr": names["to_name"][1],
        "bank": clean_bank(texts.get("from_bank")),
        "title": texts.get("title"),
        "memo": texts.get("memo"),
        "date": date,
        "ref_no": ref,
        "found": ",".join(sorted(groups)),
        "retry": ",".join(retried),                   # ช่องที่มาจากการตรวจซ้ำที่ความมั่นใจต่ำ (ควรเช็กรูปพวกนี้)
        "model": ROBOFLOW_MODEL,                      # บันทึกค่าที่ใช้รัน จะได้เทียบรอบต่อรอบได้
        "conf": CONFIDENCE,
    }
    return row, crops

ลองทดสอบ สลีปที่ไม่เคยเทรนดูสรุปที่เทรนมาอ่านได้ธนาคารที่ไม่เคยเทรนจะไม่แม่นเลย

In [ ]:
# ===== อ่านสลิปชุดใหม่ + ดูผลสด (ครบในเซลล์เดียว, ชุดนี้ไม่มีเฉลย) =====
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches

IMAGE_DIR = Path("/kaggle/input/datasets/chujikgch/slipwannatry/slip")   # สลิปชุดใหม่
MASK_FIX = False                                                       # ปิดกฎเฉพาะชุดเดิม

# 1) อ่านทุกรูป
files = sorted(p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print("พบรูป", len(files))
_trocr_errors = []
rows, t0 = [], time.time()
for path in files:
    try:
        row, _crops = read_slip(path)
    except Exception as err:
        row = {"file": path.name, "error": str(err)[:120]}
    rows.append(row)
df = pd.DataFrame(rows)
print("ใช้เวลา %.0f วินาที | error รายรูป %d | ThaiTrOCR error %d" % (
    time.time() - t0, int(df["error"].notna().sum()) if "error" in df else 0, len(_trocr_errors)))

# 2) ดูผลทีละใบ: ภาพ + กล่อง (เขียว=ยอด ฟ้า=ผู้โอน ส้ม=ผู้รับ, เส้นประ=กล่องจากการตรวจซ้ำ) + กรอบที่ตัด + ค่าที่อ่านได้
read = df.set_index("file")
COLOR = {"amount": "lime", "from_name": "deepskyblue", "to_name": "orange"}
for path in files:
    if path.name not in read.index:
        continue
    r = read.loc[path.name]
    if isinstance(r.get("error"), str):
        print(path.name, "| error:", r.get("error"))
        continue
    img = cv2.imread(str(path))
    preds, size = detect(path, RETRY_CONF)
    scale = img.shape[1] / size["width"]
    fig, ax = plt.subplots(1, 4, figsize=(16, 6), gridspec_kw={"width_ratios": [2.2, 2, 2, 2]})
    ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for p in preds:
        field = ALIAS.get(p["class"], p["class"])
        x0, y0 = (p["x"] - p["width"] / 2) * scale, (p["y"] - p["height"] / 2) * scale
        ok = p["confidence"] * 100 >= CONFIDENCE
        ax[0].add_patch(patches.Rectangle((x0, y0), p["width"] * scale, p["height"] * scale, fill=False,
                                          edgecolor=COLOR.get(field, "gray"), linewidth=2 if ok else 1, linestyle="-" if ok else "--"))
        ax[0].text(x0, y0 - 4, "%s %.0f%%" % (p["class"], p["confidence"] * 100), color="yellow", fontsize=7, bbox=dict(facecolor="black", alpha=0.6, pad=1))
    ax[0].axis("off")
    ax[0].set_title(path.name[-28:], fontsize=8)
    for i, field in enumerate(("amount", "from_name", "to_name"), 1):
        ax[i].axis("off")
        mine = sorted([p for p in preds if ALIAS.get(p["class"], p["class"]) == field], key=lambda p: -p["confidence"])
        if mine:
            ax[i].imshow(cv2.cvtColor(crop_box(img, mine[0], scale), cv2.COLOR_BGR2RGB))
            ax[i].set_title("%s %.0f%%" % (field, mine[0]["confidence"] * 100), fontsize=9)
        else:
            ax[i].set_title(field + ": no box", fontsize=9)
    plt.show()
    print(path.name)
    print("  ยอด: %s %s | ผู้โอน: %s | ผู้รับ: %s | ธนาคาร: %s | วันที่: %s | ตรวจซ้ำ: %s" % (
        r.get("amount"), ("[ธง: %s]" % r.get("amount_flag")) if r.get("amount_flag") else "",
        r.get("from_name"), r.get("to_name"), r.get("bank"), r.get("date"), r.get("retry") or "-"))

# 3) ตารางสรุป + เซฟไฟล์ (แท็บ Output)
cols = [c for c in ("file", "amount", "amount_flag", "from_name", "to_name", "bank", "title", "memo", "date", "ref_no", "retry", "error") if c in df.columns]
display(df[cols])
df[cols].to_csv("slip_live_result.csv", index=False, encoding="utf-8-sig")
print("เซฟ slip_live_result.csv")

In [ ]:
files = sorted(p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print("พบรูป", len(files))
files = files[:N_IMAGES] if N_IMAGES else files
print("รัน %d รูป | โมเดลตรวจจับ: %s | CONFIDENCE %s | ThaiTrOCR: %s | COMBINE: %s" % (len(files), ROBOFLOW_MODEL, CONFIDENCE, USE_TROCR, COMBINE))

rows, all_crops = [], {}
t0 = time.time()
for path in files:
    try:
        row, crops = read_slip(path)
    except Exception as err:
        row, crops = {"file": path.name, "error": str(err)[:120]}, {}
    rows.append(row)
    all_crops[path.name] = crops
print("ใช้เวลา %.0f วินาที" % (time.time() - t0))

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 40)

# สรุปว่าทำงานจริงไหม: error รายรูป / ช่องที่ไม่เจอกล่องหรืออ่านไม่ได้ / error ของ ThaiTrOCR
n_err = int(df["error"].notna().sum()) if "error" in df else 0
print("แถว %d | error รายรูป %d" % (len(df), n_err))
if n_err:
    display(df[df["error"].notna()][["file", "error"]].head(10))
for col in ("amount", "from_name", "to_name"):
    if col in df:
        print("%-10s ว่าง %d รูป" % (col, int(df[col].isna().sum())))
if USE_TROCR:
    print("ThaiTrOCR error %d ครั้ง%s" % (len(_trocr_errors), (" | ตัวอย่าง: " + _trocr_errors[0]) if _trocr_errors else ""))

display(df)
df.to_csv("slip_ocr_result.csv", index=False, encoding="utf-8-sig")

In [ ]:
# เทียบชื่อ: EasyOCR ล้วน / ThaiTrOCR ล้วน / รวม (COMBINE) ด้วยเกณฑ์เดียวกับ Kaggle (ชื่อละ 0.25)
# ใช้ได้ด้วยตัวเอง: ต้องมี df จากเซลล์รันหลัก (เซลล์ B) เท่านั้น
truth_path = Path("/kaggle/input/datasets/chujikgch/tuthfull/truth_full.csv")     # <- ไฟล์เฉลย
assert truth_path.exists(), "ไม่พบไฟล์เฉลย %s" % truth_path
assert "from_name_trocr" in df.columns, "df ไม่มีคอลัมน์ from_name_trocr: ต้องรันเซลล์ B ด้วย read_slip เวอร์ชันที่มี ThaiTrOCR"


def norm(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).strip()
    if s.lower() in ("none", "nan"):
        return ""
    s = s.replace("...", "").replace("ฺ", "")
    s = re.sub(r"^(?:นางสาว|นาง|นาย|น\.ส\.|mr\.?|mrs\.?|ms\.?)\s*", "", s, flags=re.I)
    return re.sub(r"[\s.*]", "", s).lower()


def _ok(pred, true):
    o, t = norm(pred), norm(true)
    return t != "" and o != "" and (o == t or o in t or t in o)


m = df.merge(pd.read_csv(truth_path), on="file", suffixes=("", "_true"))
print("รูปที่เทียบได้", len(m), "จาก", len(df))

summary = []
for col in ("from_name", "to_name"):
    _mode = COMBINE.get(col, "trocr_first") if isinstance(COMBINE, dict) else COMBINE
    for label, src in (("EasyOCR", col + "_easy"), ("ThaiTrOCR", col + "_trocr"), ("รวม " + _mode, col)):
        ok = [_ok(p, t) for p, t in zip(m[src], m[col + "_true"])]
        summary = []
for col in ("from_name", "to_name"):
    _mode = COMBINE.get(col, "trocr_first") if isinstance(COMBINE, dict) else COMBINE
    for label, src in (("EasyOCR", col + "_easy"), ("ThaiTrOCR", col + "_trocr"), ("รวม " + _mode, col)):
        ok = [_ok(p, t) for p, t in zip(m[src], m[col + "_true"])]
        summary.append({"ช่อง": col, "วิธี": label, "ถูก": sum(ok), "จาก": len(m), "คะแนนส่วนนี้(0.25)": round(0.25 * sum(ok) / len(m), 4)})
display(pd.DataFrame(summary))
if _trocr_errors:
    print("ThaiTrOCR error %d ครั้ง ตัวอย่าง: %s" % (len(_trocr_errors), _trocr_errors[0]))

# รูปที่สองวิธีให้ผลต่างกันและมีวิธีเดียวที่ถูก (ไว้ดูว่าควรเชื่อวิธีไหน)
rows_diff = []
for col in ("from_name", "to_name"):
    for _, r in m.iterrows():
        e, t = _ok(r[col + "_easy"], r[col + "_true"]), _ok(r[col + "_trocr"], r[col + "_true"])
        if e != t:
            rows_diff.append({"folder": r.get("folder"), "file": r["file"], "ช่อง": col, "ถูกโดย": "EasyOCR" if e else "ThaiTrOCR",
                              "เฉลย": r[col + "_true"], "EasyOCR": r[col + "_easy"], "ThaiTrOCR": r[col + "_trocr"]})
diff = pd.DataFrame(rows_diff)
print("ต่างกัน %d รายการ" % len(diff))
if len(diff):
    display(diff.sort_values(["ถูกโดย", "folder"]).head(60))

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    imgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
    if imgs:
        print(root, "-", len(imgs), "รูป")

In [ ]:
# ดูภาพกรอบที่ตัด เทียบกับข้อความที่อ่านได้ (ไล่ดูว่าผิดเพราะกรอบหรือเพราะ OCR)
import matplotlib.pyplot as plt

SHOW = 12
for name in list(all_crops)[:SHOW]:
    crops = all_crops[name]
    if not crops:
        continue
    fig, axes = plt.subplots(1, len(crops), figsize=(3.2 * len(crops), 2.6))
    axes = np.atleast_1d(axes)
    for ax, (field, crop) in zip(axes, crops.items()):
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(field, fontsize=9)
        ax.axis("off")
    fig.suptitle(name, fontsize=9)
    plt.show()

In [ ]:
# ดูภาพกรอบที่ตัด เทียบกับข้อความที่อ่านได้ (ไล่ดูว่าผิดเพราะกรอบหรือเพราะ OCR)
# สุ่มจากทุกโฟลเดอร์ธนาคาร โฟลเดอร์ละ PER_FOLDER ใบ (เปลี่ยน SEED เพื่อสุ่มชุดใหม่ / SEED = None = สุ่มใหม่ทุกครั้ง)
import random
import matplotlib.pyplot as plt

PER_FOLDER = 3
SEED = 1

folder_of = {p.name: p.parent.name for p in files}
by_folder = {}
for name in all_crops:
    if all_crops[name]:
        by_folder.setdefault(folder_of.get(name, "?"), []).append(name)

rng = random.Random(SEED)
read = df.set_index("file") if "file" in df else None
for folder in sorted(by_folder):
    for name in rng.sample(by_folder[folder], min(PER_FOLDER, len(by_folder[folder]))):
        crops = all_crops[name]
        fig, axes = plt.subplots(1, len(crops), figsize=(3.2 * len(crops), 2.6))
        axes = np.atleast_1d(axes)
        for ax, (field, crop) in zip(axes, crops.items()):
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            ax.set_title(field, fontsize=9)
            ax.axis("off")
        got = ""
        if read is not None and name in read.index:
            r = read.loc[name]
            got = " | อ่านได้: %s / %s / %s" % (r.get("amount"), r.get("from_name"), r.get("to_name"))
        fig.suptitle("[%s] %s%s" % (folder, name, got), fontsize=9)
        plt.show()

In [ ]:
# (ไม่บังคับ) วัดความแม่นเทียบเฉลย: ทำไฟล์ truth.csv คอลัมน์ file,amount,from_name,to_name
truth_path = Path("/kaggle/input/datasets/chujikgch/tuthfull/truth_full.csv")
if truth_path.exists():
    truth = pd.read_csv(truth_path)
    m = df.merge(truth, on="file", suffixes=("", "_true"))
    for col in ("amount", "from_name", "to_name"):
        if col + "_true" in m:
            ok = (m[col].astype(str).str.replace(" ", "") == m[col + "_true"].astype(str).str.replace(" ", ""))
            print("%-10s ถูก %d/%d" % (col, ok.sum(), len(m)))
else:
    print("ยังไม่มี truth.csv ข้ามขั้นนี้")

In [ ]:
for col in ("from_name", "to_name"):
    bad = m[m[col].astype(str).str.replace(" ", "") != m[col + "_true"].astype(str).str.replace(" ", "")]
    print("\n", col, "ผิด", len(bad), "รูป")
    display(bad[["file", col, col + "_true", "found"]])

In [ ]:
# วัดความแม่นเทียบเฉลย (truth.csv คอลัมน์ file,amount,from_name,to_name)
# นับเฉพาะแถวที่เฉลย "มีค่า" และแยก 3 แบบ: ถูก / อ่านผิด / อ่านไม่ได้เลย (ว่าง)
truth_path = Path("/kaggle/input/datasets/chujikgch/tuthfull/truth_full.csv")


def norm(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).strip()
    if s.lower() in ("none", "nan"):
        return ""
    s = s.replace("...", "").replace("\u0e3a", "")
    s = re.sub(r"^(?:นางสาว|นาง|นาย|น\.ส\.|mr\.?|mrs\.?|ms\.?)\s*", "", s, flags=re.I)
    s = re.sub(r"[\s.*]", "", s).lower()
    return s


if truth_path.exists():
    truth = pd.read_csv(truth_path)
    m = df.merge(truth, on="file", suffixes=("", "_true"))
    print("รูปที่เทียบได้", len(m), "รูป")
    for col in ("amount", "from_name", "to_name"):
        t = m[col + "_true"].map(norm)
        o = m[col].map(norm)
        has = t != ""
        if col == "amount":
            ok = has & (m[col].map(lambda v: -1 if v is None or pd.isna(v) else float(v))
                        == pd.to_numeric(m[col + "_true"], errors="coerce"))
        else:
            ok = pd.Series([tv != "" and (ov == tv or (ov != "" and (ov in tv or tv in ov)))
                            for ov, tv in zip(o, t)], index=m.index)   # ยอมรับชื่อที่ถูกสลิปตัดสั้น
        empty = has & (o == "")
        print("%-10s เฉลยมีค่า %2d | ถูก %2d | อ่านผิด %2d | อ่านไม่ได้ %2d | เฉลยว่าง(ข้าม) %d"
              % (col, has.sum(), ok.sum(), (has & ~ok & ~empty).sum(), empty.sum(), (~has).sum()))
        bad = m[has & ~ok][["file", col, col + "_true", "found"]]
        if len(bad):
            display(bad)
else:
    print("ยังไม่มี truth.csv ข้ามขั้นนี้")

In [ ]:
# ===== วินิจฉัยรูปที่ผิด: ผิดเพราะ "กล่อง" (โมเดล) หรือ "OCR"? =====
# ต้องรันเซลล์รันหลักก่อน (มีตาราง df) ใช้ detect(), ALIAS, cv2, reader จากเซลล์ก่อนหน้า
import matplotlib.pyplot as plt
import matplotlib.patches as patches

FIELD = "amount"          # amount | from_name | to_name  (ช่องที่จะตรวจ)
STATUS = "wrong,blank"    # wrong = อ่านได้แต่ผิด, blank = อ่านไม่ได้
FOLDER = None             # เช่น "scb" หรือ None = ทุกโฟลเดอร์
MAX_SHOW = 12             # แสดงกี่รูป
LOW_CONF = 5              # ความมั่นใจต่ำสุดตอนค้นหากล่องที่หลุดเกณฑ์ (%)

TRUE_COL = {"amount": "amount", "from_name": "from_name", "to_name": "to_name"}[FIELD]
CLASS_OF = {"amount": "money", "from_name": "name", "to_name": "sent"}[FIELD]   # class ใน Roboflow ของช่องนี้


def _norm(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).replace("...", "").replace("\u0e3a", "")
    s = re.sub(r"^(?:นางสาว|นาง|นาย|น\.ส\.|mr\.?|mrs\.?|ms\.?)\s*", "", s.strip(), flags=re.I)
    return re.sub(r"[\s.*]", "", s).lower()


def _status(ocr, true):
    if FIELD == "amount":
        if ocr is None or (isinstance(ocr, float) and np.isnan(ocr)):
            return "blank"
        return "ok" if abs(float(ocr) - float(true)) < 0.005 else "wrong"
    o, t = _norm(ocr), _norm(true)
    if t == "":
        return "skip"
    if o == "":
        return "blank"
    return "ok" if (o == t or o in t or t in o) else "wrong"


def _detect_low(path, conf):
    global CONFIDENCE
    keep, CONFIDENCE = CONFIDENCE, conf
    try:
        return detect(path)
    finally:
        CONFIDENCE = keep


truth_all = pd.read_csv("/kaggle/input/datasets/chujikgch/tuthfull/truth_full.csv")
paths = {p.name: p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}}
mm = df.merge(truth_all.drop(columns=["folder"], errors="ignore"), on="file", suffixes=("", "_true"))
mm["status"] = [_status(o, t) for o, t in zip(mm[FIELD], mm[TRUE_COL + "_true"])]
sel = mm[mm.status.isin(STATUS.split(","))]
if FOLDER and "folder" in sel.columns:
    sel = sel[sel.folder == FOLDER]
print("ช่อง %s: ทั้งหมด %d รูป | ที่เลือก %d รูป (แสดง %d)" % (FIELD, len(mm), len(sel), min(MAX_SHOW, len(sel))))

causes = {"box_missing": 0, "box_low_conf": 0, "ocr_error": 0}
for _, r in sel.head(MAX_SHOW).iterrows():
    path = paths.get(r.file)
    if path is None:
        continue
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    preds, size = _detect_low(path, LOW_CONF)
    scale = img.shape[1] / size["width"] if size.get("width") else 1.0
    mine = [p for p in preds if p["class"] == CLASS_OF]                       # กล่องของช่องนี้เท่านั้น
    strong = [p for p in mine if p["confidence"] * 100 >= CONFIDENCE]
    best = max(mine, key=lambda p: p["confidence"]) if mine else None

    if r.status == "wrong":
        cause = "ocr_error"                       # มีกล่องและอ่านได้ แต่ข้อความผิด
    elif strong:
        cause = "ocr_error"                       # มีกล่องผ่านเกณฑ์ แต่ OCR ได้ข้อความว่าง
    elif best is not None:
        cause = "box_low_conf"                    # มีกล่อง แต่ความมั่นใจต่ำกว่าเกณฑ์ จึงถูกทิ้ง
    else:
        cause = "box_missing"                     # โมเดลไม่เห็นช่องนี้เลย (แม้ที่ความมั่นใจต่ำสุด)
    causes[cause] += 1

    fig, ax = plt.subplots(1, 2, figsize=(11, 6), gridspec_kw={"width_ratios": [3, 1.3]})
    ax[0].imshow(img)
    for p in preds:
        field_box = p["class"] == CLASS_OF
        x0, y0 = (p["x"] - p["width"] / 2) * scale, (p["y"] - p["height"] / 2) * scale
        ax[0].add_patch(patches.Rectangle((x0, y0), p["width"] * scale, p["height"] * scale, fill=False,
                                          edgecolor=("lime" if p["confidence"] * 100 >= CONFIDENCE else "orange") if field_box else "cyan",
                                          linewidth=3 if field_box else 1))
        ax[0].text(x0, y0 - 4, "%s %.0f%%" % (p["class"], p["confidence"] * 100), color="yellow", fontsize=7,
                   bbox=dict(facecolor="black", alpha=0.6, pad=1))
    ax[0].axis("off")
    ax[0].set_title("%s | %s" % (r.get("folder", ""), r.file[-24:]), fontsize=9)
    ax[1].axis("off")
    ax[1].text(0, 1, "สาเหตุ: %s\nสถานะ: %s\n\nOCR  : %s\nจริง : %s\n\nกล่อง %s:\n%s" % (
        {"box_missing": "โมเดลไม่เจอกล่อง", "box_low_conf": "กล่องหลุดเกณฑ์ conf", "ocr_error": "OCR อ่านผิด"}[cause],
        r.status, r[FIELD], r[TRUE_COL + "_true"], CLASS_OF,
        "\n".join("  %.0f%%%s" % (p["confidence"] * 100, " (ผ่าน)" if p["confidence"] * 100 >= CONFIDENCE else " (ตก)")
                  for p in sorted(mine, key=lambda p: -p["confidence"])[:5]) or "  ไม่มี"),
        va="top", fontsize=9, family="sans-serif")
    plt.show()

print("\nสรุปสาเหตุ (ในรูปที่แสดง):", causes)
print("เขียว=กล่องช่องนี้ผ่านเกณฑ์ | ส้ม=กล่องช่องนี้ตกเกณฑ์ | ฟ้า=class อื่น")

In [ ]:
# ดูรูปที่ระบุ: กล่องทั้งหมด (ความมั่นใจต่ำสุด LOW) + กรอบชื่อที่ตัด + ข้อความจาก EasyOCR / ThaiTrOCR เทียบเฉลย
import matplotlib.pyplot as plt
import matplotlib.patches as patches

SHOW_FILES = list(dict.fromkeys(
    ["LINE_ALBUM_สลีป_261019_48.jpg", "LINE_ALBUM_สลีป_261019_60.jpg"]                          # กรุงไทย (ไม่มีทั้ง from/to)
    + ["LINE_ALBUM_19102026 _4_261019_12.jpg"]                                                    # กสิกร (label sent ขาด)
    + ["LINE_ALBUM_ออมสิน_261019_1.jpg"]                                                          # ออมสิน -> พี่ฝนกาแฟสด
    + ["LINE_ALBUM_ออมสิน_261019_%s.jpg" % n for n in (4, 5, 13, 16, 18, 21, 25, 45, 46, 49, 50)]  # ออมสิน -> ทรูมันนี่
))
LOW = 5                                    # % ความมั่นใจต่ำสุดตอนหาโชว์กล่อง
TRUTH = "/kaggle/input/datasets/chujikgch/tuthfull/truth_full.csv"

paths = {p.name: p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}}
truth = pd.read_csv(TRUTH).drop_duplicates("file").set_index("file")
summary_rows = []

for name in SHOW_FILES:
    path = paths.get(name)
    if path is None:
        print("ไม่พบไฟล์", name)
        continue
    img = cv2.imread(str(path))
    preds, size = detect(path, LOW)
    scale = img.shape[1] / size["width"]
    fig, ax = plt.subplots(1, 3, figsize=(14, 6), gridspec_kw={"width_ratios": [2, 2, 2]})
    ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for p in preds:
        field = ALIAS.get(p["class"], p["class"])
        x0, y0 = (p["x"] - p["width"] / 2) * scale, (p["y"] - p["height"] / 2) * scale
        color = ("lime" if p["confidence"] * 100 >= CONFIDENCE else "orange") if field in NAME_FIELDS else "cyan"
        ax[0].add_patch(patches.Rectangle((x0, y0), p["width"] * scale, p["height"] * scale, fill=False, edgecolor=color, linewidth=2))
        ax[0].text(x0, y0 - 4, "%s %.0f%%" % (p["class"], p["confidence"] * 100), color="yellow", fontsize=7, bbox=dict(facecolor="black", alpha=0.6, pad=1))
    ax[0].axis("off")
    ax[0].set_title(name[-22:], fontsize=8)

    print("=" * 70)
    print(name, "| เฉลย from:", truth["from_name"].get(name), "| เฉลย to:", truth["to_name"].get(name))
    row = {"file": name, "เฉลย to": truth["to_name"].get(name)}
    for i, field in enumerate(("from_name", "to_name"), 1):
        mine = sorted([p for p in preds if ALIAS.get(p["class"], p["class"]) == field], key=lambda p: -p["confidence"])
        ax[i].axis("off")
        if not mine:
            print("  %-9s ไม่มีกล่องเลย (แม้ที่ %d%%)" % (field, LOW))
            row[field + " conf%"] = None
            continue
        best = mine[0]
        crop = crop_box(img, best, scale)
        ax[i].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax[i].set_title("%s %.0f%% %s" % (field, best["confidence"] * 100, "(ผ่านเกณฑ์)" if best["confidence"] * 100 >= CONFIDENCE else "(ตกเกณฑ์)"), fontsize=9)
        easy, tro = ocr_text(crop), read_name_trocr(crop)
        print("  %-9s conf %.0f%% | EasyOCR: %r | ThaiTrOCR: %r | หลัง clean: %r / %r" % (
            field, best["confidence"] * 100, easy, tro, clean_name(easy), clean_name(tro)))
        row[field + " conf%"] = round(best["confidence"] * 100)
        if field == "to_name":
            row["EasyOCR to"], row["ThaiTrOCR to"] = clean_name(easy), clean_name(tro)
    summary_rows.append(row)
    plt.show()

# สรุป: กล่อง to_name ที่ดีที่สุดของแต่ละใบมีความมั่นใจเท่าไหร่ (ต่ำกว่า CONFIDENCE แต่ >= RETRY_CONF = เรียกคืนได้ด้วยการตรวจซ้ำ)
summ = pd.DataFrame(summary_rows)
display(summ)
retry = globals().get("RETRY_CONF", 20) or 0
c = summ["to_name conf%"]
print("to_name: ผ่านเกณฑ์ %d%% = %d ใบ | อยู่ระหว่าง %d-%d%% (ตรวจซ้ำเรียกคืนได้) = %d ใบ | ต่ำกว่า %d%% หรือไม่มีกล่อง = %d ใบ" % (
    CONFIDENCE, int((c >= CONFIDENCE).sum()), retry, CONFIDENCE, int(((c >= retry) & (c < CONFIDENCE)).sum()),
    retry, int((c.isna() | (c < retry)).sum())))